# Hierarchical Agent Teams with Azure Skills - Template

A reusable template for building a **hierarchical multi-agent system** on LangGraph where every worker agent is equipped with **Azure-backed skills** (in-process LangChain tools).

This mirrors the production design in `app/backend/agents/hierarchical_graph.py` + `agents/skills.py` of the multimodal RAG AI tutor, but is **self-contained**: the skills below are stubs you can run immediately, with `TODO` markers showing where to plug in the real Azure services.

## Topology

```
top coordinator (deterministic state-machine)
├── retrieval_team   - LLM supervisor over Azure-skill workers
│     ├── file_searcher   (file_search + azure_ai_search skills)
│     ├── graph_searcher  (graph_search skill)
│     └── notes_searcher  (notes_search + get_document skills)
├── answer_team      - answer -> verify -> followups pipeline
└── sql_team         - NL-to-SQL planner
```

Inspired by [AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation](https://arxiv.org/abs/2308.08155) (Wu et al.) and the LangGraph hierarchical-teams example.

## 1. Setup

Install dependencies and set credentials. The factory below selects **Azure OpenAI** when `AZURE_OPENAI_SERVICE` is set, otherwise plain **OpenAI**.

In [ ]:
%pip install -U langgraph langchain-core langchain-openai typing_extensions

In [ ]:
import getpass
import os


def _set_if_undefined(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Please provide your {var}: ")


# For Azure OpenAI, set these instead of OPENAI_API_KEY:
#   AZURE_OPENAI_SERVICE, AZURE_OPENAI_API_KEY,
#   AZURE_OPENAI_CHATGPT_DEPLOYMENT, AZURE_OPENAI_REASONING_DEPLOYMENT
_set_if_undefined("OPENAI_API_KEY")

## 2. Azure skills

Each skill is a LangChain `@tool` wrapping one Azure service. Workers bind only the skills they need. Here they return **stubs** so the notebook runs offline; the `TODO` comments point at the real call.

> Production version: `app/backend/agents/skills.py` (wraps `agents.tools.file_search`, `graph_search`, the Azure AI Search SDK, the notes retriever, the Verifier, and the SchemaFlow SQL pipeline).

In [ ]:
import json

from langchain_core.tools import tool


@tool
async def file_search_skill(query: str, k: int = 5) -> str:
    """Search the OpenAI file_search vector store for citation-grade evidence chunks."""
    # TODO: await agents.tools.file_search(query, k)  (OpenAI Responses file_search)
    return json.dumps([
        {"id": "doc-1", "filename": "sample.pdf", "page": 3, "score": 0.92,
         "content": f"Evidence chunk relevant to '{query}'."}
    ])


@tool
async def azure_ai_search_skill(query: str, top: int = 5) -> str:
    """Hybrid (vector + semantic) search over the primary Azure AI Search index."""
    # TODO: azure.search.documents.aio.SearchClient(...).search(query, query_type=SEMANTIC)
    return json.dumps([
        {"id": "search-1", "source_file": "sample.pdf", "page": 1, "score": 0.88,
         "content": f"AI Search hit for '{query}'."}
    ])


@tool
async def graph_search_skill(query: str, hops: int = 2, k: int = 5) -> str:
    """Search the GraphRAG knowledge graph (Cosmos Gremlin local + community summaries)."""
    # TODO: await graphrag.retriever.hybrid_search(query, hops, k)
    return json.dumps({"nodes": [], "edges": [],
                       "community_chunks": [{"id": "c1", "source": "graphrag", "text": f"Community summary for '{query}'."}]})


@tool
async def notes_search_skill(query: str, top_k: int = 5) -> str:
    """Semantic search over the notes library (Redis HNSW)."""
    # TODO: await agents.notes_search_tool.search_notes(query, top_k)
    return json.dumps({"hits": [{"id": "note-1", "content": f"Note about '{query}'."}]})


@tool
async def get_document_skill(note_id: str) -> str:
    """Fetch the full text of a note by id."""
    # TODO: await agents.notes_search_tool.get_document(note_id)
    return json.dumps({"note_id": note_id, "content": "Full note text..."})


@tool
async def verify_claim_skill(claim_sentence: str, evidence_ids: list[str]) -> str:
    """Verify whether cited evidence supports a claim. Returns {label, reason}."""
    # TODO: await agents.verifier.verify_claim(...)  (reasoning model)
    return json.dumps({"label": "supported", "reason": "evidence matches claim"})


@tool
async def sql_plan_skill(change_request: str) -> str:
    """Plan a SQL schema change with the SchemaFlow pipeline (Parse/Impact/Plan/SQL)."""
    # TODO: await SchemaFlowSQLApproach().run_for_question(change_request)
    return json.dumps({"request": change_request, "parsed": {}, "impact": {}, "plan": {}, "sql": {}})


RETRIEVAL_SKILLS = [file_search_skill, azure_ai_search_skill, graph_search_skill,
                    notes_search_skill, get_document_skill]
ANSWER_SKILLS = [verify_claim_skill]
SQL_SKILLS = [sql_plan_skill]

## 3. Chat-model factory

Selects Azure OpenAI / OpenAI and the deployment per `role` (`chat` vs `reasoning`). Mirrors `agents/_chat_model.py`.

In [ ]:
def _model_name(role: str) -> str:
    if role == "reasoning":
        return os.getenv("AZURE_OPENAI_REASONING_DEPLOYMENT",
                         os.getenv("AZURE_OPENAI_CHATGPT_DEPLOYMENT", "gpt-4.1-mini"))
    return os.getenv("AZURE_OPENAI_CHATGPT_DEPLOYMENT", os.getenv("OPENAI_CHATGPT_MODEL", "gpt-4.1-mini"))


def chat_model(role: str = "chat", temperature: float = 0.2):
    """Return a LangChain BaseChatModel for Azure / OpenAI, or None if unconfigured."""
    if os.getenv("AZURE_OPENAI_SERVICE"):
        from langchain_openai import AzureChatOpenAI

        return AzureChatOpenAI(
            azure_endpoint=f"https://{os.getenv('AZURE_OPENAI_SERVICE')}.openai.azure.com/",
            azure_deployment=_model_name(role),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-01-preview"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            temperature=temperature,
        )
    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(model=_model_name(role), temperature=temperature)
    return None


llm = chat_model("chat")
reasoning_llm = chat_model("reasoning") or llm
assert llm is not None, "Set OPENAI_API_KEY or AZURE_OPENAI_SERVICE"

## 4. Supervisor node helper

An LLM router that, given the conversation, picks the next worker or `FINISH`. Used by the retrieval team.

In [ ]:
from langgraph.graph import END, START, StateGraph, MessagesState
from langgraph.types import Command
from typing_extensions import TypedDict


def make_supervisor_node(llm, members: list[str]):
    options = ["FINISH"] + members
    system_prompt = (
        f"You are a supervisor managing these workers: {members}. Given the conversation, "
        "respond with the worker to act next. Each worker reports back. When the request is "
        "fully answered, respond with FINISH."
    )

    class Router(TypedDict):
        next: str

    def supervisor_node(state) -> Command:
        messages = [{"role": "system", "content": system_prompt}] + state["messages"]
        try:
            goto = llm.with_structured_output(Router).invoke(messages)["next"]
        except Exception:
            goto = "FINISH"
        if goto not in options:
            goto = "FINISH"
        return Command(goto=END if goto == "FINISH" else goto, update={"next": goto})

    return supervisor_node

## 5. Retrieval team

A sub-graph: an LLM supervisor over three `create_react_agent` workers, each bound to its Azure skills. This is where *"Azure skills for every agent"* lives.

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

file_agent = create_react_agent(llm, tools=[file_search_skill, azure_ai_search_skill])
graph_agent = create_react_agent(llm, tools=[graph_search_skill])
notes_agent = create_react_agent(llm, tools=[notes_search_skill, get_document_skill])


def _worker(agent, name):
    def node(state) -> Command:
        result = agent.invoke(state)
        return Command(
            update={"messages": [HumanMessage(content=result["messages"][-1].content, name=name)]},
            goto="supervisor",
        )
    return node


retrieval_builder = StateGraph(MessagesState)
retrieval_builder.add_node("supervisor", make_supervisor_node(llm, ["file_searcher", "graph_searcher", "notes_searcher"]))
retrieval_builder.add_node("file_searcher", _worker(file_agent, "file_searcher"))
retrieval_builder.add_node("graph_searcher", _worker(graph_agent, "graph_searcher"))
retrieval_builder.add_node("notes_searcher", _worker(notes_agent, "notes_searcher"))
retrieval_builder.add_edge(START, "supervisor")
retrieval_subgraph = retrieval_builder.compile()

In [ ]:
# Try the retrieval team in isolation
for step in retrieval_subgraph.stream({"messages": [("user", "What does the paper say about prompt caching?")]},
                                      {"recursion_limit": 50}):
    print(step)
    print("---")

## 6. Top graph: coordinator + teams

The top **coordinator** is a deterministic state-machine (reliable production path): route `sql` -> `sql_team`, else `retrieval_team` -> `answer_team` -> `END`.

The **answer team** and **sql team** are shown as simple LLM/stub nodes here; in production they reuse the streaming answerer + per-claim Verifier and the SchemaFlow pipeline (`app/backend/agents/hierarchical_graph.py`).

In [ ]:
from typing import Any


class State(MessagesState):
    question: str
    route: str
    evidence: Any
    answer: str
    verdict: dict


def coordinator(state: State) -> Command:
    if state.get("route") == "sql" and "verdict" not in state:
        return Command(goto="sql_team")
    if "evidence" not in state:
        return Command(goto="retrieval_team")
    if "answer" not in state:
        return Command(goto="answer_team")
    return Command(goto=END)


def retrieval_team(state: State) -> Command:
    sub = retrieval_subgraph.invoke({"messages": [HumanMessage(content=state["question"])]})
    evidence = "\n".join(m.content for m in sub["messages"] if getattr(m, "content", None))
    return Command(goto="coordinator", update={"evidence": evidence})


def answer_team(state: State) -> Command:
    # Production: agents.answerer.answer_stream + agents.verifier.verify_claim (streamed)
    prompt = f"Answer using ONLY this evidence and cite [id]:\n{state['evidence']}\n\nQuestion: {state['question']}"
    answer = llm.invoke(prompt).content
    verdict = {"unsupported_claims": 0, "retried": False}  # Verifier would fill this
    return Command(goto="coordinator", update={"answer": answer, "verdict": verdict})


def sql_team(state: State) -> Command:
    plan = sql_plan_skill.invoke({"change_request": state["question"]})
    return Command(goto="coordinator", update={"answer": plan, "verdict": {"unsupported_claims": 0, "retried": False}})


top = StateGraph(State)
top.add_node("coordinator", coordinator)
top.add_node("retrieval_team", retrieval_team)
top.add_node("answer_team", answer_team)
top.add_node("sql_team", sql_team)
top.add_edge(START, "coordinator")
graph = top.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## 7. Run it

Route a factual question through retrieval -> answer, then a SQL question through the SQL team.

In [ ]:
result = graph.invoke({"question": "What does the paper say about prompt caching?", "route": "factual"},
                      {"recursion_limit": 50})
print("ANSWER:\n", result["answer"])
print("\nVERDICT:", result["verdict"])

In [ ]:
sql_result = graph.invoke({"question": "Add a nullable `tumor_stage` column to the patients table", "route": "sql"},
                          {"recursion_limit": 50})
print(sql_result["answer"])

## 8. From template to production

To wire this into the multimodal RAG AI tutor backend:

| Template piece | Production module |
|---|---|
| Skill stubs | `app/backend/agents/skills.py` (wrap real `agents.tools.*`, AI Search SDK, notes retriever, Verifier, SchemaFlow) |
| `chat_model()` | `app/backend/agents/_chat_model.py` |
| `make_supervisor_node` + teams | `app/backend/agents/hierarchical_graph.py` (`build_hierarchical_graph`) |
| `answer_team` | reuses `agents.answerer.answer_stream` + `agents.verifier.verify_claim`, streaming **token / claim / verdict** SSE events via a LangGraph custom `StreamWriter` |
| Run loop | `approaches/hierarchical_multiagent_approach.py` - drives `graph.astream(..., stream_mode="custom")` and emits the same SSE schema as the flat path |
| Enable | set `USE_HIERARCHICAL_AGENTS=true`; `/chat` then selects the hierarchical approach |

The production answer team streams tokens and runs the Verifier **per claim** (greying out unsupported sentences in the UI). The deterministic coordinator keeps the critical path reliable while the retrieval sub-team uses a real LLM supervisor over Azure-skill agents.